In [1]:
import pandas as pd
from gensim.models import Word2Vec

import pandas as pd
import torch
from train_utils import ItemDataset, QueriesDataset, get_device, evaluate
from models import TwoTower
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch.nn.functional as F


seed = 42

d:\projects\avito_retr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Two tower + W2V

### Загрузка данных

In [2]:
# запустить после препроцесса

queries_df = pd.read_parquet('preprocessed/train_queries.parquet')
items_df = pd.read_parquet('preprocessed/train_items.parquet')

### W2V train

In [3]:
item_corpus = items_df['item_title_raw_norm']
query_corpus = queries_df['search_query_norm']

sentences = pd.concat([
    query_corpus,
    item_corpus
]).tolist()
sentences = [w.tolist() for w in sentences]

In [4]:
vec_size = 128
w2v = Word2Vec(
    sentences=sentences,
    vector_size=vec_size,
    window=5,
    min_count=2,
    sg=1,
    negative=10,
    workers=8,
    epochs=10
)

w2v.save(f'models/w2v/word2vec_{vec_size}.model')

### Train

In [3]:
w2v = Word2Vec.load('models/w2v/word2vec_128.model')

In [4]:
g = torch.Generator()
g.manual_seed(seed)

item_dataset = ItemDataset(w2v.wv, items_df=items_df)

train_df, val_df = train_test_split(queries_df, test_size=0.1, random_state=seed)
train_dataset = QueriesDataset(queries_df=train_df, item_dataset=item_dataset)
val_dataset = QueriesDataset(queries_df=val_df, item_dataset=item_dataset, mode='train')

Use Word2Vec


item_title_raw_norm: 100%|██████████| 344825/344825 [00:04<00:00, 72568.43it/s]


Use Word2Vec


search_query_norm: 100%|██████████| 447905/447905 [00:05<00:00, 77752.82it/s]


Use Word2Vec


search_query_norm: 100%|██████████| 49768/49768 [00:00<00:00, 72528.89it/s]


In [5]:
model = TwoTower()
train_dataloader = DataLoader(train_dataset, 
                              num_workers=0, 
                              shuffle=True, 
                              batch_size=1024,
                              generator=g)

Для обучения in-batch negative sampling

In [6]:
device = get_device()
model.to(device)
optimizer = torch.optim.Adam(model.parameters())

temperature = 0.1
n_epochs = 10
batches_per_epoch = 300
best_val_recall = -torch.inf
max_bad_epochs = 3

for epoch in range(n_epochs):
    model.train()   
    batch_iter = iter(train_dataloader)
    pbar = tqdm(range(batches_per_epoch))
    loss_sum = 0
    for batch_idx in pbar:
        q_emb, i_emb, context, item_ids, _, _ = [tensor.to(device) for tensor in next(batch_iter)]
        
        q, i = model((q_emb, i_emb, context))
        logits = (q @ i.T) / temperature
        
        same_item = item_ids[:, None] == item_ids[None, :]
        diag = torch.arange(logits.size(0), device=device)
        same_item[diag, diag] = False

        logits[same_item] = -torch.inf
        targets = torch.arange(logits.size(0), device=device)
        loss = F.cross_entropy(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        pbar.set_description(f"Epoch {epoch} --- loss: {loss_sum / (batch_idx + 1)} | ")

    recall = evaluate(queries_dataset=val_dataset, items_dataset=item_dataset, model=model, device=device, k=50, filters=False)
    print(recall)
    if best_val_recall > recall['Recall@50']:
        max_bad_epochs -= 1
    else:
        best_val_recall = recall['Recall@50']
        torch.save(model.state_dict(), "models/w2v/two_tower_w2v_128.pth")
    if max_bad_epochs == 0:
        break

Epoch 0 --- loss: 5.48961895942688 | : 100%|██████████| 300/300 [00:12<00:00, 24.37it/s]  
Queries: 100%|██████████| 778/778 [00:05<00:00, 140.53it/s]


{'Recall@50': 0.084130364893104}


Epoch 1 --- loss: 2.948384826183319 | : 100%|██████████| 300/300 [00:12<00:00, 23.95it/s] 
Queries: 100%|██████████| 778/778 [00:04<00:00, 157.93it/s]


{'Recall@50': 0.10583105610030542}


Epoch 2 --- loss: 2.741247631708781 | : 100%|██████████| 300/300 [00:11<00:00, 25.10it/s] 
Queries: 100%|██████████| 778/778 [00:05<00:00, 143.16it/s]


...

{'Recall@50': 0.11147725446069763}


Epoch 5 --- loss: 2.5959996501604716 | : 100%|██████████| 300/300 [00:12<00:00, 23.34it/s]
Queries: 100%|██████████| 778/778 [00:05<00:00, 147.96it/s]


{'Recall@50': 0.11513422279376306}


Epoch 6 --- loss: 2.5689361921946205 | : 100%|██████████| 300/300 [00:13<00:00, 22.93it/s]
Queries: 100%|██████████| 778/778 [00:05<00:00, 150.22it/s]

{'Recall@50': 0.11423002732679634}


In [34]:
recall = evaluate(queries_dataset=val_dataset, items_dataset=item_dataset, model=model, device=device, k=50, filters=False)
recall_filtered = evaluate(queries_dataset=val_dataset, items_dataset=item_dataset, model=model, device=device, k=50, filters=True)

Queries: 100%|██████████| 778/778 [00:29<00:00, 26.31it/s]


In [35]:
print(recall)
print(recall_filtered)

{'Recall@50': 0.11443095965278895}
{'Recall@50': 0.7414000964475165}


### gen ans

In [38]:
queries_df_test = pd.read_parquet('preprocessed/benchmark_queries_preprocessed_w2v.parquet')
items_df_test = pd.read_parquet('preprocessed/benchmark_items_preprocessed_w2v.parquet')
w2v = Word2Vec.load('models/w2v/word2vec_128.model')

In [39]:
device = get_device()
g = torch.Generator()
g.manual_seed(seed)

item_dataset_test = ItemDataset(w2v.wv, items_df=items_df_test)
test_dataset = QueriesDataset(queries_df=queries_df_test, item_dataset=item_dataset_test, mode='test')

model = TwoTower()
model.load_state_dict(torch.load("models/w2v/two_tower_w2v_128.pth"))
model.to(device)

Use Word2Vec


item_title_raw_norm: 100%|██████████| 189212/189212 [00:02<00:00, 73156.44it/s]


Use Word2Vec


search_query_norm: 100%|██████████| 2452/2452 [00:00<00:00, 68096.20it/s]


TwoTower(
  (q_tower): QTower(
    (dense): Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
      (3): ReLU()
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): ReLU()
    )
  )
  (i_tower): ITower(
    (context_proj): Sequential(
      (0): Linear(in_features=5, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=128, bias=True)
      (3): ReLU()
    )
    (dense): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=64, bias=True)
      (3): ReLU()
    )
  )
)

In [40]:
ans = evaluate(queries_dataset=test_dataset, items_dataset=item_dataset_test, model=model, device=device, k=50, filters=True)

Queries: 100%|██████████| 39/39 [00:00<00:00, 41.12it/s]


In [41]:
answer = pd.DataFrame({
    'query_id': queries_df_test['query_id'],                                   # строки
    'answer': [' '.join(top50) for top50 in item_dataset_test.item_ids[torch.cat(ans, dim=0)]],     # строки
})
answer.to_csv('ans/answer_w2v.csv', index=False)

Recall@50: 0.658399